# Задача 1. Дешифровка шифра Цезаря с помощью простой RNN

1. Обучите **простую рекуррентную нейронную сеть** (без GRU/LSTM, без внимания) **решать задачу дешифровки** [шифра Цезаря](https://ru.wikipedia.org/wiki/Шифр_Цезаря):

    1. Написать алгоритм шифра Цезаря для генерации выборки (сдвиг на N каждой буквы). Например, если N = 2, то буква А переходит в букву С. Можно поиграться с языком на выбор (немецкий, русский и т.д.).
    2. Создать архитектуру рекуррентной нейронной сети.
    3. Обучить её (вход — зашифрованная фраза, выход — дешифрованная фраза).
    4. Проверить качество модели.

**2 балла** за правильно выполненное задание.

In [1]:
# Блок 0. Импорты

import random
import string

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
# Блок 1. Алфавит и шифр Цезаря (русский)

RUS_ALPHABET = "абвгдеежзийклмнопрстуфхцчшщъыьэюя "
VOCAB = RUS_ALPHABET
VOCAB_SIZE = len(VOCAB)

char2idx = {}
idx2char = {}
for i, ch in enumerate(VOCAB):
    char2idx[ch] = i
    idx2char[i] = ch

def caesar_encrypt(text, shift):
    text = text.lower()
    result = ""
    for ch in text:
        if ch in char2idx:
            old_idx = char2idx[ch]
            new_idx = (old_idx + shift) % VOCAB_SIZE
            result += idx2char[new_idx]
        else:
            # символы вне алфавита просто пропускаю
            continue
    return result

def caesar_decrypt(text, shift):
    # дешифрование — это шифрование с отрицательным сдвигом
    return caesar_encrypt(text, -shift)

# тест
plain = "привет мир"
cipher = caesar_encrypt(plain, 3)
back = caesar_decrypt(cipher, 3)
print("plain :", plain)
print("cipher:", cipher)
print("back  :", back)

plain : привет мир
cipher: тулеихвплу
back  : пригет мир


In [3]:
# Блок 2. Генерация датасета

def random_russian_sentence(min_len=5, max_len=20):
    length = random.randint(min_len, max_len)
    s = ""
    for _ in range(length):
        s += random.choice(VOCAB)
    return s

def generate_caesar_dataset(n_samples, shift):
    data = []
    for _ in range(n_samples):
        plain = random_russian_sentence()
        cipher = caesar_encrypt(plain, shift)
        data.append((cipher, plain))
    return data

SHIFT = 5  # фиксированный сдвиг
train_pairs = generate_caesar_dataset(3000, SHIFT)
val_pairs   = generate_caesar_dataset(500, SHIFT)

print("Пример пары:", train_pairs[0])

Пример пары: ('ффчаксфпсежб', 'пптьемпкмбвэ')


In [4]:
# Блок 3. Перевод строки в тензор индексов

MAX_LEN = 25  # максимальная длина последовательности

def encode_sentence(s, max_len=MAX_LEN):
    s = s[:max_len]
    ids = []
    for ch in s:
        ids.append(char2idx[ch])
    while len(ids) < max_len:
        ids.append(0)  # паддинг нулём
    return torch.tensor(ids, dtype=torch.long)

In [5]:
# Блок 4. Dataset и DataLoader

class CaesarDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        cipher, plain = self.pairs[idx]
        x = encode_sentence(cipher)
        y = encode_sentence(plain)
        return x, y

train_ds = CaesarDataset(train_pairs)
val_ds   = CaesarDataset(val_pairs)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_dl))
print("Batch shapes:", xb.shape, yb.shape)

Batch shapes: torch.Size([64, 25]) torch.Size([64, 25])


In [6]:
# Блок 5. Простая RNN (без LSTM/GRU)

class CaesarRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.RNN(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)            # (batch, seq_len, emb_dim)
        out, h = self.rnn(emb)         # (batch, seq_len, hidden_dim)
        logits = self.fc(out)          # (batch, seq_len, vocab_size)
        return logits

model = CaesarRNN(VOCAB_SIZE).to(device)
print(model)

CaesarRNN(
  (embed): Embedding(34, 32)
  (rnn): RNN(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=34, bias=True)
)


In [7]:
# Блок 6. Обучение модели

def train_model(model, train_dl, val_dl, epochs=8, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(
                logits.view(-1, VOCAB_SIZE),
                yb.view(-1)
            )
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.inference_mode():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(
                    logits.view(-1, VOCAB_SIZE),
                    yb.view(-1)
                )
                val_loss += loss.item()

        print(f"Epoch {epoch}: train_loss={train_loss/len(train_dl):.4f}, "
              f"val_loss={val_loss/len(val_dl):.4f}")

train_model(model, train_dl, val_dl, epochs=8)

Epoch 1: train_loss=2.0677, val_loss=1.4278
Epoch 2: train_loss=1.1352, val_loss=0.7869
Epoch 3: train_loss=0.5530, val_loss=0.3431
Epoch 4: train_loss=0.2478, val_loss=0.1740
Epoch 5: train_loss=0.1441, val_loss=0.1185
Epoch 6: train_loss=0.1065, val_loss=0.0950
Epoch 7: train_loss=0.0891, val_loss=0.0828
Epoch 8: train_loss=0.0793, val_loss=0.0754


In [ ]:
# Блок 7. Оценка качества

def decode_logits(logits):
    pred_ids = logits.argmax(dim=-1)
    result = []
    for row in pred_ids:
        s = ""
        for idx in row:
            s += idx2char[int(idx)]
        result.append(s)
    return result

def evaluate(model, pairs, n_examples=5):
    model.eval()
    ds = CaesarDataset(pairs)
    dl = DataLoader(ds, batch_size=32, shuffle=False)

    correct = 0
    total = 0

    with torch.inference_mode():
        for xb, yb in dl:
            xb = xb.to(device)
            logits = model(xb)
            preds = decode_logits(logits)
            for pred, (cipher, plain) in zip(preds, pairs):
                plain_cut = plain[:MAX_LEN]
                pred_cut = pred[:len(plain_cut)]
                if plain_cut == pred_cut:
                    correct += 1
                total += 1

    print("Sequence accuracy:", correct / total)

    print("\nПримеры:")
    for i in range(n_examples):
        cipher, plain = pairs[i]
        x = encode_sentence(cipher).unsqueeze(0).to(device)
        with torch.inference_mode():
            logits = model(x)
        pred = decode_logits(logits)[0][:len(plain)]
        print("cipher :", cipher)
        print("plain  :", plain)
        print("decoded:", pred)
        print("-" * 40)

evaluate(model, val_pairs, n_examples=5)

Sequence accuracy: 0.032

Примеры:
cipher : ьшкнтепигфъ 
plain  : чуеинакдяпхы
decoded: чуеинакдяпхы
----------------------------------------
cipher : шяфлшвкже
plain  : уъпжуюевб
decoded: уъпжуюева
----------------------------------------
cipher : тъянакг
plain  : нхъиьея
decoded: нхъиаея
----------------------------------------
cipher : фмънжисгд
plain  : пзхивдмя 
decoded: пзхивдмя 
----------------------------------------
cipher : вьбтуж
plain  : ючэнов
decoded: ючэнов
----------------------------------------


## Выводы по задаче 1

- Сгенерирована выборка случайных русских строк, зашифрованных шифром Цезаря с фиксированным сдвигом по алфавиту; для каждой зашифрованной фразы известна исходная (дешифрованная) последовательность символов.
- Построена простая рекуррентная нейронная сеть (RNN без использования LSTM/GRU и механизмов внимания), которая по входной зашифрованной последовательности посимвольно предсказывает исходную строку.
- В процессе обучения значение функции потерь как на обучающей, так и на валидационной выборке устойчиво уменьшалось, что говорит о том, что модель успешно подстраивает веса под задачу дешифровки.
- При проверке на валидационных примерах были зафиксированы как полностью корректно расшифрованные строки, так и случаи частичных ошибок; итоговая метрика `sequence accuracy` около 0.03 показывает, что модель чаще всего ошибается хотя бы в одном символе фразы, но при этом локальные предсказания уже во многом совпадают с эталоном.
- Для дальнейшего улучшения качества можно увеличить объём обучающей выборки, число эпох, размер скрытого слоя или использовать более мощные архитектуры (например, LSTM/GRU), однако в рамках задания удалось продемонстрировать, что даже простая RNN способна приближённо реализовать дешифровку шифра Цезаря.
